# 3. Warmup phase retrieval only
Independent HAPRE → ER warmup on your corrected holograms. There is **no physical projection** and no joint outer loop. The red `supportmask.png` must match the centered object grid produced by notebook 2; red means allowed support, including object and reference holes.

The solver receives the already prepared grid with crop=0 and binning=1. This keeps the saved warmup fields on the same grid as their holograms. The largest connected support component is the default material region; inspect its overlay. All exact inputs, fields, diagnostics, and mode geometry are saved together.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
if HERE.name != "permanent_magnets":
    HERE = HERE / "permanent_magnets"
REPO = HERE.parent
sys.path.insert(0, str(REPO))
from permanent_magnets import workflow

from library import phase_retrieval_universal as universal

In [ ]:
PROCESSED = Path("/path/to/processed")  # Edit: same output folder in all four notebooks.
INPUT = PROCESSED / "02_holograms.h5"
OUTPUT = PROCESSED / "03_warmup.h5"
SUPPORT_PNG = HERE / "masks" / "supportmask.png"  # Red = allowed object/support pixels.
MODES = [1, 2]  # [1] for a single mode; 2 enlarges the second support.
WARMUP_ITERATIONS = [320, 50]  # HAPRE, ER.
FFT_WORKERS = 1

# Remaining explicit solver settings (same convention as the controls above).
PROJECTION_MODEL = "none"
OUTER_ITERATIONS = 0
WARMUP_MODE = ["HAPRE", "ER"]
WARMUP_BETA_MODE = ["arctan", "const"]
WARMUP_ALPHA_ZERO = .5
WARMUP_TV_FREQ = 10
WARMUP_START_FROM_FIRST = False
STARTIMAGE_SCALE_FIT = "through_origin"
RECENTER_MODAL_SUPPORTS = True
MODE_SUPPORT_CENTER = "image"
MODE_INITIALIZATION = "support_fft"
MODE_INITIALIZATION_SEED = 0
CROP = 0
BINNING = 1
ROI = None
HOLOGRAM_INTENSITY_CUTOFF_VMIN = -1
PARTIAL_COHERENCE = False
FIT_ILLUMINATION_INTENSITY = False
ALIGN_GLOBAL_PHASE = False
FIT_APERTURE_ILLUMINATION = False
FINAL_PROJECTION_RELAXATION = 0.
FINAL_FOURIER_CONSTRAINT = False
CONSTRAIN_NONPHYSICAL_MODES_COMMON = False
RANDOM_SEED = 0

# Keep CROP=0, BINNING=1, ROI=None: the holograms were prepared in notebook 2.
# PROJECTION_MODEL="none" and OUTER_ITERATIONS=0 keep this stage warmup-only.


In [ ]:
with h5py.File(INPUT, "r") as h:
    holograms = h["holograms"][()]
    mask_pixel = h["mask_pixel"][()].astype(bool)
    scan_indices = h["scan_indices"][()]
    energies = h["energy_eV"][()]
    polarizations = h["polarization"][()]
support = workflow.red_mask(SUPPORT_PNG, holograms.shape[-2:])
material = universal.largest_support_component(support).astype(float)
state_labels = [f"scan_{index}" for index in scan_indices]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(support, cmap="gray"); axes[0].set_title("Painted support (object + references)")
axes[1].imshow(material, cmap="gray"); axes[1].set_title("Largest component: material region")
fth_preview = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(np.where(mask_pixel[0], 0, holograms[0]))))
axes[2].imshow(np.log1p(abs(fth_preview)))
axes[2].contour(support, levels=[.5], colors="red", linewidths=.5)
axes[2].set_title("FTH preview with support overlay")
plt.show()
# The library requires a recipe dictionary; editable controls stay as variables above.
recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    projection_model=PROJECTION_MODEL, outer_iterations=OUTER_ITERATIONS,
    warmup_mode=WARMUP_MODE, warmup_Nit=WARMUP_ITERATIONS,
    warmup_beta_mode=WARMUP_BETA_MODE, warmup_alpha_zero=WARMUP_ALPHA_ZERO, warmup_TV_freq=WARMUP_TV_FREQ,
    warmup_start_from_first=WARMUP_START_FROM_FIRST, startimage_scale_fit=STARTIMAGE_SCALE_FIT,
    modes=MODES, recenter_modal_supports=RECENTER_MODAL_SUPPORTS, mode_support_center=MODE_SUPPORT_CENTER,
    mode_initialization=MODE_INITIALIZATION, mode_initialization_seed=MODE_INITIALIZATION_SEED,
    crop=CROP, binning=BINNING, roi=ROI, hologram_intensity_cutoff_vmin=HOLOGRAM_INTENSITY_CUTOFF_VMIN,
    partial_coherence=PARTIAL_COHERENCE, fit_illumination_intensity=FIT_ILLUMINATION_INTENSITY,
    align_global_phase=ALIGN_GLOBAL_PHASE, fit_aperture_illumination=FIT_APERTURE_ILLUMINATION,
    final_projection_relaxation=FINAL_PROJECTION_RELAXATION, final_fourier_constraint=FINAL_FOURIER_CONSTRAINT,
    constrain_nonphysical_modes_common=CONSTRAIN_NONPHYSICAL_MODES_COMMON, fft_workers=FFT_WORKERS, random_seed=RANDOM_SEED,
)
_, warmup, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, support, state_labels=state_labels,
    energy_labels=energies, polarization_coefficients=polarizations,
    illumination_labels=["beam0"]*len(state_labels), universal_recipe=recipe,
    phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel,
)
assert warmup.shape[-2:] == holograms.shape[-2:]

In [ ]:
objects, intensity_residual = workflow.field_diagnostics(warmup, holograms, bsmasks, state_labels)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT, "w") as h:
    h.attrs["source_file"] = str(INPUT.resolve())
    h.attrs["support_png"] = str(SUPPORT_PNG.resolve())
    h.attrs["crop"] = 0
    h.attrs["binning"] = 1
    h.attrs["partial_coherence"] = False
    for name, array in [("warmup_fields", warmup), ("objects", objects),
                        ("holograms", holograms), ("mask_pixel", mask_pixel),
                        ("warmup_bsmasks", bsmasks), ("supportmask", support),
                        ("material_thickness", material), ("modes", MODES),
                        ("warmup_iterations", WARMUP_ITERATIONS), ("scan_indices", scan_indices),
                        ("energy_eV", energies), ("polarization", polarizations),
                        ("intensity_residual", intensity_residual)]:
        h.create_dataset(name, data=array, compression="gzip")
    h.create_dataset("state_labels", data=state_labels, dtype=h5py.string_dtype())
    workflow.save_tree(h.create_group("components"), components)
    workflow.save_tree(h.create_group("errors"), errors)
    workflow.save_tree(h.create_group("recipe"), recipe)
print("Saved", OUTPUT)